In [ ]:
import random
import csv

import math

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches

import textwrap


In [ ]:
def parse_bbox(bbox_str):
    if pd.isna(bbox_str) or bbox_str == '':
        return None
    return tuple(map(int, bbox_str.split(',')))

In [ ]:
def visualize_csv_boxes_grid(csv_path, num_samples=35, boxes_per_row=5, random_seed=42):
    df = pd.read_csv(csv_path)
    sampled_rows = df.sample(n=num_samples, random_state=random_seed)

    num_cols = boxes_per_row
    num_rows = math.ceil(num_samples / num_cols)

    fig, axes = plt.subplots(num_rows, num_cols, figsize=(boxes_per_row * 4, num_rows * 4))
    axes = axes.flatten()  # easy indexing even if last row is incomplete

    for ax, (idx, row) in zip(axes, sampled_rows.iterrows()):
        # Read the boxes
        boxes = []
        labels = []
        for i in range(1, 5):
            obj_col = f'obj{i}'
            bbox_col = f'bbox{i}'
            obj_name = row[obj_col]
            bbox = parse_bbox(row[bbox_col])
            if obj_name and isinstance(obj_name, str) and bbox:
                boxes.append(bbox)
                labels.append(obj_name)

        # Plot this row
        ax.set_xlim(0, 512)
        ax.set_ylim(0, 512)
        ax.invert_yaxis()

        for i, box in enumerate(boxes):
            x1, y1, x3, y3 = box
            width = x3 - x1
            height = y3 - y1
            rect = patches.Rectangle((x1, y1), width, height, linewidth=2, edgecolor='r', facecolor='none')
            ax.add_patch(rect)

            # Label
            ax.text(x1 + 3, y1 - 5, labels[i], color='blue', fontsize=8)

        ax.set_title(f'ID {row["id"]}\n{row["prompt"]}\n{row["category"]}', fontsize=10)
        ax.grid(True)

    # Hide unused axes (if num_samples is not multiple of boxes_per_row)
    for ax in axes[num_samples:]:
        ax.axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
visualize_csv_boxes_grid('openSet.csv', num_samples=35, boxes_per_row=5, random_seed=42)

In [ ]:
def visualize_csv_boxes_by_ids(csv_path, selected_ids, boxes_per_row=4, output_path=None):
    import pandas as pd
    import matplotlib.pyplot as plt
    import matplotlib.patches as patches
    import math

    def parse_bbox(bbox_str):
        try:
            return list(map(int, bbox_str.split(',')))
        except:
            return None

    df = pd.read_csv(csv_path)

    padded_ids = [str(id_) for id_ in selected_ids]
    selected_rows = df[df['id'].astype(str).isin(padded_ids)]

    if selected_rows.empty:
        print("⚠️ No matching IDs found.")
        return

    num_samples = len(selected_rows)
    num_cols = boxes_per_row
    num_rows = math.ceil(num_samples / num_cols)

    fig, axes = plt.subplots(num_rows, num_cols, figsize=(boxes_per_row * 4, num_rows * 4), squeeze=False)
    axes = axes.flatten()

    for ax, (_, row) in zip(axes, selected_rows.iterrows()):
        boxes, labels = [], []
        for i in range(1, 5):
            obj_col, bbox_col = f'obj{i}', f'bbox{i}'
            obj_name = row[obj_col]
            bbox = parse_bbox(row[bbox_col])
            if obj_name and isinstance(obj_name, str) and bbox:
                boxes.append(bbox)
                labels.append(obj_name)

        ax.set_xlim(0, 512)
        ax.set_ylim(0, 512)
        ax.invert_yaxis()

        for i, box in enumerate(boxes):
            x1, y1, x3, y3 = box
            rect = patches.Rectangle((x1, y1), x3 - x1, y3 - y1,
                                     linewidth=2, edgecolor='r', facecolor='none')
            ax.add_patch(rect)
            ax.text(x1 + 3, y1 - 5, labels[i], color='blue', fontsize=8)


        id_text = f"ID {row['id']}"
        prompt_text = "\n".join(textwrap.wrap(row["prompt"], width=50))  # adjust width
        category_text = row["category"]

        title_text = f"{id_text}\n{prompt_text}\n{category_text}"

        ax.set_title(title_text, fontsize=10)
        ax.grid(True)

    # remove extra axes entirely
    for ax in axes[num_samples:]:
        fig.delaxes(ax)

    plt.tight_layout()

    if output_path:
        plt.savefig(output_path, bbox_inches='tight')

    plt.show()


In [ ]:
# Visualize specific prompts (e.g., ID 0, 5, 18, and 29)
visualize_csv_boxes_by_ids("fullNewDataset.csv", selected_ids=[183], output_path="2359.png")

In [ ]:
visualize_csv_boxes_by_ids("fullNewDataset.csv", selected_ids=[2158,2219,2336,2517], output_path="sb_boxes.pdf")

In [ ]:
visualize_csv_boxes_by_ids("fullNewDataset.csv", selected_ids=[2008], output_path="over_boxes.pdf")

In [ ]:
visualize_csv_boxes_by_ids("fullNewDataset.csv", selected_ids=[783], output_path="cb_boxes.pdf")

In [ ]:
visualize_csv_boxes_by_ids("fullNewDataset.csv", selected_ids=[1058,1254,1303,1454], output_path="ab_boxes.pdf")


In [ ]:
visualize_csv_boxes_by_ids("fullNewDataset.csv", selected_ids=[2731], output_path="or_boxes.pdf")

In [ ]:
visualize_csv_boxes_by_ids("fullNewDataset.csv", selected_ids=[2951], output_path="cc_boxes.pdf")

In [ ]:
visualize_csv_boxes_by_ids("openSet.csv", selected_ids=[4,385,1499,2471,3033,3210,3275,3318], output_path="open_boxes.pdf")
